In [ ]:
import subprocess, time, os, re, requests, shutil

# ── Config ───────────────────────────────────────────────
MODEL       = "EVA-abliterated-TIES-Qwen2.5-14B-i1-GGUF:Q6_K"
PUBLIC_NAME = "character1"
API_KEY     = "sk-colab-local"
PROXY_PORT  = 4000

def sh(cmd): return subprocess.run(cmd, shell=True)
def bg(cmd, log): return subprocess.Popen(cmd, shell=True, stdout=open(log,"w"), stderr=subprocess.STDOUT)
def wait(url, t=180, name=""):
    for _ in range(t):
        try:
            if requests.get(url, timeout=2).status_code < 500:
                print(f"  ok  {name} ready"); return True
        except: pass
        time.sleep(1)
    print(f"  !!  {name} NOT ready"); return False

sh("pkill -f 'ollama serve'; pkill -f litellm; pkill -f cloudflared"); time.sleep(2)

print(">> [1/6] Installing Ollama (GitHub .tar.zst — the method that works on Colab) ...")
sh("apt-get install -y -qq zstd")
sh("rm -f /content/ollama.tar.zst")
sh("curl -fL -o /content/ollama.tar.zst "
   "https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst")
sh("ls -la /content/ollama.tar.zst")           # cek: harusnya ratusan MB, bukan 0
sh("tar --zstd -xf /content/ollama.tar.zst -C /usr")
OLLAMA = shutil.which("ollama") or ("/usr/bin/ollama" if os.path.exists("/usr/bin/ollama") else None)
print("   ollama binary:", OLLAMA or "STILL NOT FOUND")

if not OLLAMA:
    print("\n  STOP: masih gagal. Tempel output 'ls -la' di atas ke saya.")
else:
    print(">> [2/6] Installing LiteLLM + cloudflared ...")
    sh("pip -q install 'litellm[proxy]'")
    sh("curl -fL -o /usr/bin/cloudflared "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
       "&& chmod +x /usr/bin/cloudflared")

    print(">> [3/6] Starting Ollama ...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    bg(f"{OLLAMA} serve", "/content/ollama.log")
    wait("http://localhost:11434", name="ollama")

    print(f">> [4/6] Pulling model {MODEL} (a few minutes) ...")
    sh(f"{OLLAMA} pull {MODEL}")

    print(">> [5/6] Starting LiteLLM proxy ...")
    open("/content/litellm.yaml","w").write(f"""
model_list:
  - model_name: {PUBLIC_NAME}
    litellm_params:
      model: ollama/{MODEL}
      api_base: http://localhost:11434
general_settings:
  master_key: {API_KEY}
litellm_settings:
  drop_params: true
""")
    bg(f"litellm --config /content/litellm.yaml --port {PROXY_PORT} --host 0.0.0.0", "/content/litellm.log")
    if not wait(f"http://localhost:{PROXY_PORT}/health/liveliness", name="litellm"):
        print("   --- litellm.log ---"); print(open("/content/litellm.log").read()[-1200:])

    print(">> [6/6] Opening Cloudflare tunnel ...")
    bg(f"cloudflared tunnel --url http://localhost:{PROXY_PORT} --no-autoupdate", "/content/cloudflared.log")
    url = None
    for _ in range(40):
        try:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
            if m: url = m.group(0); break
        except: pass
        time.sleep(1)
    print("\n" + "="*60)
    if url:
        print("  READY — put these in your local .env:\n")
        print(f"  BASE_URL={url}")
        print(f"  LLM_API_KEY={API_KEY}")
        print(f"  MODEL_NAME={PUBLIC_NAME}")
    else:
        print("  Tunnel URL not found — check /content/cloudflared.log")
    print("="*60)
    print("\n  Keep this cell RUNNING while you use the app.")